### **Section 11: Response Models**

We are now entering Response Control & Logic. Up until now, we have focused on how data *enters* the API. Now, we focus on how data *leaves* the API and how we communicate with the Client.<br>
A **Response Model** is a tool that allows you to define exactly what data your API should return to the user. It is one of the most important security features in FastAPI.

1. Why do we need it?<br>
Imagine you have a `User` model in your database that contains a `username`, `email`, and a `hashed_password`. If you simply return the User object, the API will send the password to the internet.<br>
A **Response Model** acts as a "filter." You define a Pydantic model with only the fields you want the user to see, and FastAPI handles the filtering for you.

2. The Syntax<br>
You define the response model inside the path decorator using the `response_model` parameter.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, EmailStr
from typing import List, Optional

app = FastAPI()

# This is our internal data "contract"
class UserIn(BaseModel):
    username: str
    password: str  # We receive the password
    email: EmailStr
    full_name: Optional[str] = None

# This is our public "filter"
class UserOut(BaseModel):
    username: str
    email: EmailStr
    full_name: Optional[str] = None
    # Notice: 'password' is NOT here!

@app.post("/user/", response_model=UserOut)
async def create_user(user: UserIn):
    # In a real app, you'd save the user to a database here
    return user 


# **Technical Result:** Even though the function returns the full `user` object (which includes the password), 
# FastAPI will look at `response_model=UserOut`, strip away the password field, and only send the username, email, and name to the client.

### **Section 12: HTTP Status Codes**

When an API responds, it doesn't just send data; it sends a **Status Code**. This is a 3-digit number that tells the Client the result of their request.<br>
1. Common Status Codes to Know
* **200 OK**: The request was successful. (Default for GET)
* **201 Created**: The request was successful and a new resource was created. (Standard for POST)
* **400 Bad Request**: The client sent something wrong.
* **401 Unauthorized**: The client needs to log in.
* **404 Not Found**: The resource (like a user ID) doesn't exist.
* **500 Internal Server Error**: Your Python code crashed.

2. Setting the Status Code in FastAPI<br>
You can set a default status code for a route in the decorator:

In [83]:
from fastapi import status

@app.post("/items/", status_code=status.HTTP_201_CREATED)
async def create_item(name: str):
    return {"message": "Item created", "name": name}

### **Section 13: Error Handling**

What happens when something goes wrong? For example, a user asks for `/items/999`, but item 999 doesn't exist. You shouldn't just return an empty dictionary; you should return a proper error.<br>
In FastAPI, we use the `HTTPException` class to stop the execution and send an error back to the client.<br>
1. Detailed Working Example:

In [84]:
from fastapi import FastAPI, HTTPException

app = FastAPI()

items = {"foo": "The Boring Tool", "bar": "The Bar Tender"}

@app.get("/items/{item_id}")
async def read_item(item_id: str):
    if item_id not in items:
        # We 'raise' the exception to stop the function immediately
        raise HTTPException(
            status_code=404, 
            detail="Item not found",
            headers={"X-Error": "There goes my error"} # Optional extra info
        )
    return {"item": items[item_id]}

2. Technical Breakdown ⚙️

* **`raise`**: This keyword stops the function. No code after this line will run.
* **`status_code=404`**: Tells the browser/client exactly what kind of error it is.
* **`detail`**: This becomes the "message" inside the JSON response that the user sees.


**Connecting the Logic**<br>
Response models, status codes, and error handling work together to make your API professional and safe.<br>
Imagine you are building a **Login API**.<br>
1. The user sends a Request Body (Section 10).
2. You check the database.
3. If the password is wrong, you **raise an HTTPException** with a **401 status code**.
4. If the password is correct, you return the User data using a **Response Model** to ensure the hashed password is never seen.

### **Section 14: Dependency Injection**

In programming, **Dependency Injection** is a fancy way of saying: *"Don't make your function do everything. Let it ask for what it needs."*<br>
1. The Problem: The "Messy" Function<br>
Imagine you are building an API for a secret club. Every time someone wants to see a page, you have to:<br>
1. Check the URL for a secret code.
2. Verify if that code is "12345".
3. If it's wrong, tell them to go away.
**The WRONG way (Repeating yourself):**

In [85]:
from fastapi import FastAPI, HTTPException

app = FastAPI()

@app.get("/secret-room")
async def secret_room(password: str):
    # This logic is trapped inside this function
    if password != "12345":
        raise HTTPException(status_code=401, detail="Wrong password")
    return {"message": "Welcome to the secret room!"}

@app.get("/vault")
async def vault(password: str):
    # You have to copy-paste the SAME logic here!
    if password != "12345":
        raise HTTPException(status_code=401, detail="Wrong password")
    return {"money": "1,000,000"}

**Why this is bad:** If you change the password to "54321", you have to find every single function and change it manually. If you miss one, your security is broken.

2. The Solution: The "Dependency" Function<br>
Instead of copy-pasting, we create a **Dependency**. This is just a normal function that handles the "requirement" (the dependency).<br>
**Step 1: Create the requirement function**

In [86]:
from fastapi import Header, HTTPException

# This function defines the "rule"
async def verify_password(secret_password: str):
    if secret_password != "12345":
        raise HTTPException(status_code=401, detail="Unauthorized")
    return secret_password # We return the value if it's correct


**Step 2: Use `Depends` to "Inject" it**<br>
Now, we tell our routes: "You cannot run unless `verify_password` says it's okay."

In [87]:
from fastapi import Depends, FastAPI

app = FastAPI()

# We import the 'Depends' class from fastapi
@app.get("/secret-room")
async def secret_room(pw: str = Depends(verify_password)):
    # Look at 'pw'. FastAPI runs 'verify_password' first. 
    # Whatever that function returns is put into 'pw'.
    return {"message": "Welcome!", "used_password": pw}

@app.get("/vault")
async def vault(pw: str = Depends(verify_password)):
    return {"money": "1,000,000"}

3. How the "Injection" actually happens (The Magic)<br>
When a user visits `/vault?secret_password=12345`, here is the step-by-step "handshake" FastAPI performs:<br>
1. **The Interception:** FastAPI sees `Depends(verify_password)`. It pauses the `vault` function.
2. **The Sub-request:** FastAPI looks at `verify_password`. It sees that this function needs a variable called `secret_password`.
3. **The Extraction:** FastAPI automatically looks at the URL, finds `secret_password=12345`, and gives it to the `verify_password` function.
4. **The Validation:** `verify_password` runs. If it raises an error, the `vault` function **never even starts**.
5. **The Injection:** If the password is correct, the result is "injected" into the variable `pw`, and the `vault` function finally runs.


4. Real-World Power: Database Sessions<br>
The most common use for this in FastAPI is managing **Database Connections**. You don't want to keep a database connection open forever (it wastes memory). You want to open it when a request starts and close it the second it ends.
**The Professional Pattern:**

In [88]:
def get_db():
    db = "Open Connection to Database" # Imagine this is real code
    try:
        yield db  # The 'yield' keyword sends 'db' to the route
    finally:
        print("Closing Database...") # This runs AFTER the route finishes
        # Close connection here
        
# By using `Depends(get_db)`, you ensure that your database is handled safely every single time without writing `try/finally` in every route.

### **Section 15: Database Integration**

Since the last section focused on how we *inject* logic (Dependencies), it is the perfect time to move to Database Integration.<br>
This is the most critical part of any real-world API because most applications need to remember data even after the server restarts. To do this in Python, we use a library called **SQLAlchemy**.<br>

1. What is an ORM? (The Translator)<br>
A database (like PostgreSQL or SQLite) speaks **SQL** (Structured Query Language). Python speaks **Objects** (Classes).<br>
An **ORM (Object-Relational Mapper)** like SQLAlchemy acts as a translator. It allows you to write Python code to create tables and save data, and it automatically converts that into SQL commands for the database.

| In Python (The Model) | In SQL (The Table) |
| --- | --- |
| `class User(Base):` | `CREATE TABLE users ...` |
| `name = Column(String)` | `name VARCHAR` |
| `new_user = User(name="Alice")` | `INSERT INTO users (name) VALUES ('Alice')` |

2. The 5-Step Setup<br>

To connect FastAPI to a database, you need to set up these five pieces in your project. We usually put these in a file called `database.py`.<br>
1. The Database URL<br>
This tells Python where the database file is located.

In [89]:
SQLALCHEMY_DATABASE_URL = "sqlite:///./sql_app.db"

2. The Engine<br>
The "Engine" is the actual connection manager that talks to the database file.

In [90]:
from sqlalchemy import create_engine

engine = create_engine(
    SQLALCHEMY_DATABASE_URL,
    connect_args={"check_same_thread": False}  # Required for SQLite with FastAPI
)

3. The SessionLocal<br>
This is a "factory." Every time a user visits your API, this factory creates a unique "Session" (a temporary connection) just for that person.

In [91]:
from sqlalchemy.orm import sessionmaker

SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)

4. The Base Class<br>
This is the "parent" class. Every table we create later will inherit from this so SQLAlchemy knows they belong to the same database.

In [92]:
from sqlalchemy.orm import declarative_base

Base = declarative_base()

5. The Database Dependency (The "Yield" Pattern)<br>
Remember **Section 14**? We use a dependency to give each route a database session.

In [93]:
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

3. Creating a Table (The Model)<br>
Now we define what our data looks like. Let's create a **User** table in a file called `models.py`.<br>

In [94]:
from sqlalchemy.orm import Mapped, mapped_column
from sqlalchemy import String, Boolean, Integer
from database import Base

class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, index=True)
    email: Mapped[str] = mapped_column(String, unique=True, index=True)
    hashed_password: Mapped[str] = mapped_column(String)
    is_active: Mapped[bool] = mapped_column(Boolean, default=True)

ModuleNotFoundError: No module named 'database'

4. Using it in a FastAPI Route<br>
Now we bring it all together. We use **Depends** to get the database session and then use standard Python code to save a user.

In [ ]:
from fastapi import FastAPI, Depends
from sqlalchemy.orm import Session
from . import models
from .database import engine, get_db

# This line creates the tables in your database file automatically
models.Base.metadata.create_all(bind=engine)

app = FastAPI()

@app.post("/users/")
def create_user(email: str, db: Session = Depends(get_db)):
    # 1. Create a Python object based on our Model
    new_user = models.User(email=email, hashed_password="fake-password-123")
    
    # 2. Tell SQLAlchemy to prepare to save it
    db.add(new_user)
    
    # 3. "Commit" tells the database to save it permanently
    db.commit()
    
    # 4. Refresh pulls the new ID from the database into our object
    db.refresh(new_user)
    
    return new_user

**Why this is better than "Standard" SQL**
1. **Safety:** SQLAlchemy automatically protects you from "SQL Injection" (hackers trying to run malicious code in your database).
2. **Flexibility:** If you start with a simple **SQLite** database and later want to move to a powerful **PostgreSQL** database, you only have to change **one line** (the Database URL). The rest of your code stays exactly the same.

To fetch a user from the database, we need to combine everything we’ve learned: **Path Parameters** (to get the ID from the URL), **Database Models** (to know what a "User" looks like), and **Dependency Injection** (to get the database connection).<br>
Let's write a "Read" route that finds a specific user by their ID.<br>
**The "Read User" Route**<br>
We will add this to the code we started in the previous section.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy.orm import Session
from . import models
from .database import get_db

app = FastAPI()

@app.get("/users/{user_id}")
def read_user(user_id: int, db: Session = Depends(get_db)):
    # 1. Ask the database to 'query' the User table
    # 2. .filter tells SQL "WHERE id = user_id"
    # 3. .first() gives us the first matching result (or None)
    user = db.query(models.User).filter(models.User.id == user_id).first()
    
    # 4. Error Handling: If the database returned None, the user doesn't exist
    if user is None:
        raise HTTPException(status_code=404, detail="User not found in our records")
    
    # 5. Return the user object (FastAPI converts it to JSON automatically)
    return user

**Breaking Down the Technical Logic**<br>
Let's look at exactly what is happening under the hood when a user visits `http://127.0.0.1:8000/users/5`.

1. **The Extraction**: FastAPI sees the `{user_id}` in the path and extracts the number `5`.
2. **The Injection**: Because of `Depends(get_db)`, FastAPI opens a connection to your database file and hands it to the function as the variable `db`.
3. **The Query**: `db.query(models.User)` tells SQLAlchemy: *"Prepare a SELECT statement for the 'users' table."*
4. **The Filter**: `.filter(models.User.id == user_id)` adds the specific condition: *"Only give me the row where the ID column matches 5."*
5. **The Execution**: `.first()` actually runs the command.
* If a row is found, it creates a Python object filled with that user's data.
* If no row is found, it returns `None`.
6. **The Closing**: Because we used `yield` in our `get_db` function earlier, the database connection **automatically closes** as soon as the `return` or `raise` line finishes.


**Common Database Operations Table**

When you are writing "Read" logic, you will use these SQLAlchemy methods constantly:

| Python Code | SQL Equivalent | Result |
| --- | --- | --- |
| `.query(User).all()` | `SELECT * FROM users;` | Returns a **List** of every user. |
| `.query(User).first()` | `SELECT * FROM users LIMIT 1;` | Returns the **First** user found. |
| `.filter(User.id == 5)` | `WHERE id = 5` | **Filters** the results. |
| `.offset(10).limit(5)` | `OFFSET 10 LIMIT 5` | Used for **Pagination** (skipping rows). |

### **Section 16: Security & Authentication**

We are now entering Security & Authentication<br>
This is where we transition from a public API to a professional, secure system. In this section, we will learn how to verify who a user is (Authentication) and ensure they only see the data they are allowed to see (Authorization).<br>
1. The Standard: OAuth2 and JWT<br>
FastAPI uses **OAuth2** as the "flow" and **JWT (JSON Web Tokens)** as the "pass."<br>
Think of it like a **Hotel**:
1. **Authentication (The Front Desk):** You show your ID and provide your reservation (Login).
2. **The Token (The Key Card):** The receptionist gives you a plastic key card (The JWT).
3. **Authorization (Entering the Room):** You don't show your ID to every door. You just tap your key card. The door "decodes" the card to see if you have access to that specific room.

2. What is a JWT (JSON Web Token)?<br>
A JWT is a long string of text that is cryptographically signed. It is **not** encrypted (anyone can read it), but it is **signed** so that if even one letter is changed, the server will know it has been tampered with.<br>
It has three parts separated by dots:<br>
1. **Header:** Tells the server what kind of token it is.
2. **Payload:** Contains the data, like `user_id: 5` and `expires: 30mins`.
3. **Signature:** Created using a **SECRET_KEY** that only your server knows.


Step 1: Password Hashing<br>
You must **never** store a user's password in plain text in your database. If a hacker steals your database, they get everyone's password. Instead, we "Hash" it.<br>
A hash is a one-way mathematical function. You can turn "password123" into a hash, but you cannot turn the hash back into "password123."<br>
**The Logic:**<br>
* **Sign up:** User sends "pass123" -> Server hashes it to "$2b$12$..." -> Server saves the hash.
* **Login:** User sends "pass123" -> Server hashes the new input and compares it to the hash in the database. If they match, the password is correct.<br>
We use the library `passlib` to handle this.

Step 2: The Login Route (Issuing the Token)<br>
To start the security flow, we need a route where the user "exchanges" their username/password for a Token.

In [ ]:
from datetime import datetime, timedelta
from jose import jwt
from fastapi import Depends, FastAPI, HTTPException, status
from fastapi.security import OAuth2PasswordRequestForm

# 1. Configuration
SECRET_KEY = "my_ultra_secret_key" # Keep this hidden!
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

app = FastAPI()

@app.post("/token")
async def login(form_data: OAuth2PasswordRequestForm = Depends()):
    # In a real app, you would check 'form_data.username' against your Database
    user_in_db = {"username": "jdoe", "hashed_password": "..."} 
    
    if not form_data.password == "secret123": # Simplified check
        raise HTTPException(status_code=400, detail="Incorrect username or password")

    # 2. Create the JWT Data
    expire = datetime.utcnow() + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    to_encode = {"sub": form_data.username, "exp": expire}
    
    # 3. Sign the Token
    encoded_jwt = jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
    
    return {"access_token": encoded_jwt, "token_type": "bearer"}

Step 3: Protecting Routes (The Key Card Check)<br>
Now that the user has a token, how do we use it to protect a route? We use the `OAuth2PasswordBearer` dependency.

In [ ]:
from fastapi.security import OAuth2PasswordBearer

# This tells FastAPI: "Look for the token in the Authorization Header"
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

@app.get("/users/me")
async def read_users_me(token: str = Depends(oauth2_scheme)):
    # 1. This function will only run if the 'token' exists
    # 2. You would then decode the token to find out WHICH user this is
    return {"token_received": token, "user": "Verified User"}

How to Test Security in Swagger (`/docs`)<br>
FastAPI makes testing security incredibly easy:<br>

1. Go to `http://127.0.0.1:8000/docs`.
2. You will see a green **"Authorize"** button at the top.
3. Click it, enter "jdoe" and "secret123".
4. FastAPI will automatically perform the POST request to `/token`, get the JWT, and **save it in your browser's memory**.
5. Now, every time you click "Execute" on a protected route, FastAPI automatically sends that token for you!

### **Section 17: Background Tasks**

We are now entering Background Tasks. In a professional API, you often have tasks that take a long time to finish, such as:
* Sending a confirmation email.
* Processing a large image or video.
* Syncing data with an external service.

If you make the user wait for these tasks to finish, your API will feel "laggy" and slow. **Background Tasks** allow you to trigger these actions and immediately tell the user "Success!", while the server continues working on the heavy task in the background.

1. The Problem: The "Waiting" User<br>
Imagine a user signs up for your app. Your code needs to:
* Save the user to the database (Fast: 0.01s).
* Connect to an email server and send a Welcome Email (Slow: 3.0s).
* Return a "Welcome" message to the user.

**Without Background Tasks:** The user stares at a loading spinner for **3.1 seconds** before they see the "Welcome" page. This is a bad user experience.

2. The Solution: `BackgroundTasks`<br
FastAPI provides a built-in class called `BackgroundTasks`. It allows you to "schedule" a function to run **after** the response has been sent to the client.

* How to define the background function<br>
This is just a regular Python function. It doesn't even need to be `async` (though it can be).

In [ ]:
def send_welcome_email(email: str, message: str):
    # This simulates the slow process of sending an email
    import time
    time.sleep(5) 
    print(f"Email sent to {email}: {message}")

2. How to "Schedule" it in your Route<br>
You add `BackgroundTasks` as a parameter to your route. FastAPI handles the "Injection" (Section 14) for you.

In [ ]:
from fastapi import BackgroundTasks, FastAPI

app = FastAPI()

@app.post("/signup/{email}")
async def signup(email: str, background_tasks: BackgroundTasks):
    # 1. Logic to save user to DB goes here (Fast)
    
    # 2. Schedule the slow task
    background_tasks.add_task(send_welcome_email, email, "Welcome to our API!")
    
    # 3. Return the response IMMEDIATELY
    return {"message": "Signup successful! Check your email in a few moments."}

3. Technical Breakdown: What happens under the hood? ⚙️

* **The Request**: The user hits the `/signup` route.
* **The Addition**: `background_tasks.add_task` puts your function and its arguments into a "To-Do List" inside FastAPI.
* **The Response**: FastAPI sends the JSON `{"message": "Signup successful!"}` back to the user's phone or browser immediately.
* **The Execution**: Only **after** the connection with the user is closed, the server looks at its "To-Do List" and starts running `send_welcome_email`.

4. When to use Background Tasks vs. Celery ⚖️

FastAPI's built-in `BackgroundTasks` is great for simple things like emails or small logs. However, if you are doing **extremely heavy** work (like training an AI model or processing thousands of videos), you should use a more powerful tool like **Celery**.

| Feature | FastAPI BackgroundTasks | Celery / Redis |
| --- | --- | --- |
| **Setup** | Built-in, zero setup. | Requires a "Broker" (Redis/RabbitMQ). |
| **Execution** | Runs on the same server. | Runs on separate "Worker" servers. |
| **Reliability** | If the server crashes, the task is lost. | Tasks are saved and retried if they fail. |
| **Best For** | Sending emails, simple logging. | Video encoding, heavy data analysis. |